In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

orders = spark.table("novacart_dev.silver.slv_orders")

In [0]:
# Dimension: Dates (calendar dimension)
# Get date range from orders
date_range = orders.agg(
    F.min("order_date").alias("min_date"),
    F.max("order_date").alias("max_date")
).collect()[0]

# Generate date dimension
from datetime import datetime, timedelta
import pandas as pd

start_date = date_range['min_date']
end_date = date_range['max_date']

# Generate date sequence
date_list = []
current = start_date
while current <= end_date:
    date_list.append({
        'date_key': int(current.strftime('%Y%m%d')),
        'full_date': current,
        'year': current.year,
        'quarter': (current.month - 1) // 3 + 1,
        'month': current.month,
        'month_name': current.strftime('%B'),
        'day': current.day,
        'day_of_week': current.weekday() + 1,  # 1=Monday, 7=Sunday
        'day_name': current.strftime('%A'),
        'is_weekend': 1 if current.weekday() >= 5 else 0
    })
    current += timedelta(days=1)

# Create dimension
dim_dates = spark.createDataFrame(pd.DataFrame(date_list))

print(f"dim_dates: {dim_dates.count():,} dates")
print(f"Date range: {start_date} to {end_date}")
display(dim_dates.limit(5))

In [0]:
# Write dim_dates to Gold layer
dim_dates.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("novacart_dev.gold.dim_dates")

print("✓ dim_dates written to novacart_dev.gold.dim_dates")